In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
df = pd.read_excel(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\datatest.xlsx")
df.head()

,id,video
0,1,https://www.instagram.com/reel/DM2DYuURXAS/?ig...
1,2,https://www.instagram.com/reel/DMpUrCKxQj5/?ig...
2,3,https://www.instagram.com/reel/DKoahGuRUGB/?ig...
3,4,https://www.instagram.com/reel/DKZVA8DJq89/?ig...
4,5,https://www.instagram.com/reel/DLtizEnyaOn/?ig...


In [17]:
import pandas as pd
import os
import yt_dlp
import time
from pathlib import Path

class SimpleVideoDownloader:
    def __init__(self, output_dir="downloaded_videos"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def download_video(self, url, video_id):
        """Download video saja menggunakan yt-dlp dengan cookie authentication"""
        try:
            # Nama file output
            filename = f"{video_id}"
            
            # Konfigurasi yt-dlp - FOKUS HANYA VIDEO
            ydl_opts = {
                'outtmpl': os.path.join(self.output_dir, f'{filename}.%(ext)s'),
                'format': 'best',  # Video quality terbaik <= 720p
                # Alternative browsers:
                # 'cookiesfrombrowser': ('firefox',),
                # 'cookiesfrombrowser': ('safari',),
                
                # Headers untuk bypass detection
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
                },
                
                # Retry settings
                'retries': 3,
                'fragment_retries': 3,
                
                # Sleep to avoid rate limiting
                'sleep_interval': 2,
                'max_sleep_interval': 5,
                
                # Suppress output untuk cleaner logs
                'quiet': False,
                'no_warnings': False,
            }
            
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                # Download video
                ydl.download([url])
                
                # Cari file yang berhasil didownload
                for ext in ['mp4', 'webm', 'mkv']:
                    potential_file = os.path.join(self.output_dir, f"{filename}.{ext}")
                    if os.path.exists(potential_file):
                        return potential_file
                
                return None
                
        except Exception as e:
            print(f"❌ Error downloading video {video_id}: {str(e)}")
            return None
    
    def process_csv_batch(self, df, batch_num):
        """Proses satu batch video"""
        results = []
        total_videos = len(df)
        
        print(f"\n🔄 Processing Batch {batch_num} - {total_videos} videos")
        print("="*50)
        
        for index, row in df.iterrows():
            video_id = row['id']
            url = row['video']
            
            print(f"📥 [{index+1}/{total_videos}] Downloading video {video_id}")
            
            # Download video
            video_path = self.download_video(url, video_id)
            
            if video_path and os.path.exists(video_path):
                file_size = os.path.getsize(video_path) / (1024*1024)  # MB
                print(f"✅ Success: {os.path.basename(video_path)} ({file_size:.1f} MB)")
                
                result = {
                    'id': video_id,
                    'video_file': os.path.basename(video_path),
                    'video_path': video_path,
                    'original_url': url,
                    'status': 'success',
                    'file_size_mb': round(file_size, 1)
                }
            else:
                print(f"❌ Failed: Could not download video {video_id}")
                result = {
                    'id': video_id,
                    'video_file': None,
                    'video_path': None,
                    'original_url': url,
                    'status': 'failed',
                    'file_size_mb': 0
                }
            
            results.append(result)
            
            # Delay antar download
            print(f"⏳ Waiting 3 seconds...")
            time.sleep(3)
        
        return pd.DataFrame(results)

def download_instagram_videos(csv_file_path, batch_size=25, start_from=0):
    """
    Main function untuk download video Instagram
    
    Args:
        csv_file_path (str): Path ke file CSV
        batch_size (int): Jumlah video per batch
        start_from (int): Index mulai (untuk resume)
    """
    
    print("🚀 SIMPLE INSTAGRAM VIDEO DOWNLOADER")
    print("="*50)
    
    try:
        # Baca CSV
        print(f"📖 Reading CSV: {csv_file_path}")
        df = pd.read_csv(csv_file_path)
        
        print(f"📊 Found columns: {df.columns.tolist()}")
        print(f"📈 Total rows: {len(df)}")
        print(f"🎯 Sample data:")
        print(df.head())
        
        # Validasi kolom
        required_cols = ['id', 'video']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"❌ Missing columns: {missing_cols}")
            return
        
        # Filter data untuk diproses
        df_to_process = df.iloc[start_from:].reset_index(drop=True)
        print(f"\n🎯 Processing {len(df_to_process)} rows starting from index {start_from}")
        
        # Inisialisasi downloader
        downloader = SimpleVideoDownloader()
        
        # Process in batches
        all_results = []
        
        for batch_start in range(0, len(df_to_process), batch_size):
            batch_end = min(batch_start + batch_size, len(df_to_process))
            batch_df = df_to_process.iloc[batch_start:batch_end].reset_index(drop=True)
            
            batch_num = (batch_start // batch_size) + 1
            
            # Proses batch
            batch_results = downloader.process_csv_batch(batch_df, batch_num)
            all_results.append(batch_results)
            
            # Simpan hasil batch
            batch_filename = f'batch_videos_{batch_num}.csv'
            batch_results.to_csv(batch_filename, index=False)
            
            # Summary batch
            successful = len(batch_results[batch_results['status'] == 'success'])
            total_size = batch_results['file_size_mb'].sum()
            
            print(f"\n📊 Batch {batch_num} Summary:")
            print(f"   ✅ Downloaded: {successful}/{len(batch_df)} videos")
            print(f"   📊 Success rate: {successful/len(batch_df)*100:.1f}%")
            print(f"   💾 Total size: {total_size:.1f} MB")
            print(f"   💾 Saved to: {batch_filename}")
            
            # Break jika ini batch terakhir
            if batch_end >= len(df_to_process):
                break
                
        # Gabungkan semua hasil
        if all_results:
            final_results = pd.concat(all_results, ignore_index=True)
            
            # Simpan hasil final
            final_filename = f'final_videos_{start_from}_{start_from + len(df_to_process) - 1}.csv'
            final_results.to_csv(final_filename, index=False)
            
            # Summary keseluruhan
            total_successful = len(final_results[final_results['status'] == 'success'])
            total_size = final_results['file_size_mb'].sum()
            
            print(f"\n🎉 FINAL SUMMARY")
            print("="*50)
            print(f"📊 Total processed: {len(final_results)} videos")
            print(f"✅ Successfully downloaded: {total_successful} videos")
            print(f"📈 Overall success rate: {total_successful/len(final_results)*100:.1f}%")
            print(f"💾 Total downloaded size: {total_size:.1f} MB")
            print(f"📁 Videos saved in: downloaded_videos/")
            print(f"📄 Results saved to: {final_filename}")
            
            
            return final_results
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None

# CARA PENGGUNAAN
if __name__ == "__main__":
    # SETUP - Ganti dengan path file CSV Anda
    csv_file = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\datatest - datatest.csv"
    
    print("🔧 SETUP INSTRUCTIONS:")
    print("1. Make sure you're logged into Instagram in Chrome browser")
    print("2. Install required packages: pip install yt-dlp pandas")
    print("3. The script will use cookies from Chrome automatically")
    print("\n" + "="*50)
    
    # Konfirmasi untuk melanjutkan
    confirm = input("Ready to start downloading? (y/n): ").lower().strip()
    
    if confirm == 'y':
        # JALANKAN DOWNLOAD
        results = download_instagram_videos(
            csv_file_path=csv_file,
            batch_size=20,      # Download 20 video per batch
            start_from=0        # Mulai dari index 0
        )
        
        if results is not None:
            print("🎉 Download completed successfully!")
        else:
            print("❌ Download failed. Check error messages above.")
    else:
        print("👋 Download cancelled.")

# TROUBLESHOOTING GUIDE
"""
🔧 TROUBLESHOOTING:

1. LOGIN ERROR:
   - Login ke Instagram di browser Chrome
   - Refresh halaman Instagram
   - Jalankan script lagi

2. RATE LIMIT:
   - Kurangi batch_size ke 10-15
   - Tingkatkan sleep time di script
   - Gunakan VPN dengan IP berbeda

3. SLOW DOWNLOAD:
   - Kurangi batch_size
   - Pastikan koneksi internet stabil
   - Close aplikasi lain yang pakai bandwidth

4. RESUME DOWNLOAD:
   - Lihat batch terakhir yang berhasil
   - Set start_from ke index yang sesuai
   
5. BROWSER COOKIE ISSUES:
   - Coba browser lain: ubah 'chrome' ke 'firefox' atau 'safari'
   - Clear browser cache dan login ulang
"""

🔧 SETUP INSTRUCTIONS:
1. Make sure you're logged into Instagram in Chrome browser
2. Install required packages: pip install yt-dlp pandas
3. The script will use cookies from Chrome automatically



Ready to start downloading? (y/n):  y


🚀 SIMPLE INSTAGRAM VIDEO DOWNLOADER
📖 Reading CSV: C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\datatest - datatest.csv
📊 Found columns: ['id', 'video']
📈 Total rows: 200
🎯 Sample data:
   id                                              video
0   1  https://www.instagram.com/reel/DM2DYuURXAS/?ig...
1   2  https://www.instagram.com/reel/DMpUrCKxQj5/?ig...
2   3  https://www.instagram.com/reel/DKoahGuRUGB/?ig...
3   4  https://www.instagram.com/reel/DKZVA8DJq89/?ig...
4   5  https://www.instagram.com/reel/DLtizEnyaOn/?ig...

🎯 Processing 200 rows starting from index 0

🔄 Processing Batch 1 - 20 videos
📥 [1/20] Downloading video 1
[Instagram] Extracting URL: https://www.instagram.com/reel/DM2DYuURXAS/?igsh=Z3lzNDZtdDhxeWxn
[Instagram] DM2DYuURXAS: Setting up session
[Instagram] DM2DYuURXAS: Downloading JSON metadata
[info] DM2DYuURXAS: Downloading 1 format(s): 8
[download] Sleeping 4.33 seconds ...
[download] Destination: downloaded_videos\1.mp4
[download] 100% of   10.

[Instagram] C3pEeaaS7oG: Downloading JSON metadata
[info] C3pEeaaS7oG: Downloading 1 format(s): 3
[download] Sleeping 4.88 seconds ...
[download] Destination: downloaded_videos\48.mp4
[download] 100% of   16.30MiB in 00:00:01 at 14.39MiB/s  
✅ Success: 48.mp4 (16.3 MB)
⏳ Waiting 3 seconds...
📥 [9/20] Downloading video 49
[Instagram] Extracting URL: https://www.instagram.com/reel/C3ZwNdMS3F8/?igsh=dm5xY2l0cG5rcDF3
[Instagram] C3ZwNdMS3F8: Setting up session
[Instagram] C3ZwNdMS3F8: Downloading JSON metadata
[info] C3ZwNdMS3F8: Downloading 1 format(s): 3
[download] Sleeping 2.28 seconds ...
[download] Destination: downloaded_videos\49.mp4
[download] 100% of   28.62MiB in 00:00:01 at 16.88MiB/s  
✅ Success: 49.mp4 (28.6 MB)
⏳ Waiting 3 seconds...
📥 [10/20] Downloading video 50
[Instagram] Extracting URL: https://www.instagram.com/reel/DHuzjz-yZON/?igsh=MWViYjBtbDg4bGEwcQ==
[Instagram] DHuzjz-yZON: Setting up session
[Instagram] DHuzjz-yZON: Downloading JSON metadata
[info] DHuzjz-yZON: Do

❌ Error downloading video 107: expected string or bytes-like object, got 'bool'
❌ Failed: Could not download video 107
⏳ Waiting 3 seconds...
📥 [8/20] Downloading video 108
[GoogleDrive] Extracting URL: https://drive.google.com/file/d/1PMDBY0C5oekMhFDm2KcR2jLjdgonZ2Bi/view?usp=sharing
[GoogleDrive] 1PMDBY0C5oekMhFDm2KcR2jLjdgonZ2Bi: Downloading video webpage
[GoogleDrive] 1PMDBY0C5oekMhFDm2KcR2jLjdgonZ2Bi: Requesting source file
[info] 1PMDBY0C5oekMhFDm2KcR2jLjdgonZ2Bi: Downloading 1 format(s): source
[download] Sleeping 4.02 seconds ...
[download] Destination: downloaded_videos\108.mp4
[download] 100% of    7.97MiB in 00:00:02 at 3.31MiB/s   
✅ Success: 108.mp4 (8.0 MB)
⏳ Waiting 3 seconds...
📥 [9/20] Downloading video 109
[GoogleDrive] Extracting URL: https://drive.google.com/file/d/1ElvqnnKfMLU8pZyKKs_CMxwGts2O770v/view?usp=sharing
[GoogleDrive] 1ElvqnnKfMLU8pZyKKs_CMxwGts2O770v: Downloading video webpage
[GoogleDrive] 1ElvqnnKfMLU8pZyKKs_CMxwGts2O770v: Requesting source file
[info]

[Instagram] DM8PAMHPsh4: Downloading JSON metadata
[info] DM8PAMHPsh4: Downloading 1 format(s): 4
[download] Sleeping 2.95 seconds ...
[download] Destination: downloaded_videos\121.mp4
[download] 100% of   39.02MiB in 00:00:02 at 18.75MiB/s  
✅ Success: 121.mp4 (39.0 MB)
⏳ Waiting 3 seconds...
📥 [2/20] Downloading video 122
[Instagram] Extracting URL: https://www.instagram.com/reel/DM7I5rNp3au/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] DM7I5rNp3au: Setting up session
[Instagram] DM7I5rNp3au: Downloading JSON metadata
[info] DM7I5rNp3au: Downloading 1 format(s): 4
[download] Sleeping 3.48 seconds ...
[download] Destination: downloaded_videos\122.mp4
[download] 100% of   10.25MiB in 00:00:00 at 10.52MiB/s  
✅ Success: 122.mp4 (10.3 MB)
⏳ Waiting 3 seconds...
📥 [3/20] Downloading video 123
[Instagram] Extracting URL: https://www.instagram.com/reel/DM6mD8jyIV6/?utm_source=ig_web_copy_link
[Instagram] DM6mD8jyIV6: Setting up session
[Instagram] DM6mD8jyIV6: Downloading J

ERROR: [Instagram] DM0ChALScuv: Restricted Video: You must be 18 years old or over to see this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies


❌ Error downloading video 166: ERROR: [Instagram] DM0ChALScuv: Restricted Video: You must be 18 years old or over to see this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies
❌ Failed: Could not download video 166
⏳ Waiting 3 seconds...
📥 [7/20] Downloading video 167
[Instagram] Extracting URL: https://www.instagram.com/reel/DIyUyveT0SA/?igsh=ZTEyaGRjc2dpZzhi
[Instagram] DIyUyveT0SA: Setting up session
[Instagram] DIyUyveT0SA: Downloading JSON metadata
[info] DIyUyveT0SA: Downloading 1 format(s): 1
[download] Sleeping 2.80 seconds ...
[download] Destination: downloaded_videos\167.mp4
[download] 100% of    4.58MiB in 00:00:00 at 11.05MiB/s  
✅ Success: 167.mp4 (4.6 MB)
⏳ Waiting 3 seconds...
📥 [8/20] Downloading video 168
[Instagram] Extracting URL: https://www.instagram.com/reel/DNIfBeBzlo_/?igsh=YTg3cW1zY3Eyd3dy
[Instagram] DNIfBeBzlo_: Setting up sess

"\n🔧 TROUBLESHOOTING:\n\n1. LOGIN ERROR:\n   - Login ke Instagram di browser Chrome\n   - Refresh halaman Instagram\n   - Jalankan script lagi\n\n2. RATE LIMIT:\n   - Kurangi batch_size ke 10-15\n   - Tingkatkan sleep time di script\n   - Gunakan VPN dengan IP berbeda\n\n3. SLOW DOWNLOAD:\n   - Kurangi batch_size\n   - Pastikan koneksi internet stabil\n   - Close aplikasi lain yang pakai bandwidth\n\n4. RESUME DOWNLOAD:\n   - Lihat batch terakhir yang berhasil\n   - Set start_from ke index yang sesuai\n   \n5. BROWSER COOKIE ISSUES:\n   - Coba browser lain: ubah 'chrome' ke 'firefox' atau 'safari'\n   - Clear browser cache dan login ulang\n"

In [19]:
import os
import yt_dlp
import time

class SimpleVideoDownloader:
    def __init__(self, output_dir="downloaded_videos"):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def download_video(self, url, idx):
        """Download video langsung dari link"""
        try:
            filename = f"video_{idx}"
            
            ydl_opts = {
                'outtmpl': os.path.join(self.output_dir, f'{filename}.%(ext)s'),
                'format': 'best',  
                'http_headers': {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                                  'Chrome/120.0.0.0 Safari/537.36'
                },
                'retries': 3,
                'fragment_retries': 3,
                'sleep_interval': 2,
                'max_sleep_interval': 5,
                'quiet': False,
                'no_warnings': False,
            }
            
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
                
                for ext in ['mp4', 'webm', 'mkv']:
                    potential_file = os.path.join(self.output_dir, f"{filename}.{ext}")
                    if os.path.exists(potential_file):
                        return potential_file
                return None
        except Exception as e:
            print(f"❌ Error downloading {url}: {str(e)}")
            return None

def download_from_links(links):
    downloader = SimpleVideoDownloader()
    results = []

    print(f"🚀 Downloading {len(links)} videos...\n")

    for idx, url in enumerate(links, 1):
        print(f"📥 [{idx}/{len(links)}] {url}")
        video_path = downloader.download_video(url, idx)

        if video_path and os.path.exists(video_path):
            size = os.path.getsize(video_path) / (1024 * 1024)
            print(f"✅ Success: {os.path.basename(video_path)} ({size:.1f} MB)")
            results.append({"url": url, "status": "success", "file": video_path})
        else:
            print("❌ Failed")
            results.append({"url": url, "status": "failed", "file": None})

        print("⏳ Waiting 3s...\n")
        time.sleep(3)
    
    print("\n🎉 DONE")
    return results


if __name__ == "__main__":
    # Masukkan link di sini langsung
    video_links = [
        'https://www.instagram.com/reel/DLuZNKOzFQ_/?igsh=M284bXZ6a3ppZHF5',
        'https://www.instagram.com/reel/DM0ChALScuv/?igsh=MWphajVsYzFvMWcydg%3D%3D'
    ]
    
    results = download_from_links(video_links)


🚀 Downloading 2 videos...

📥 [1/2] https://www.instagram.com/reel/DLuZNKOzFQ_/?igsh=M284bXZ6a3ppZHF5
[Instagram] Extracting URL: https://www.instagram.com/reel/DLuZNKOzFQ_/?igsh=M284bXZ6a3ppZHF5
[Instagram] DLuZNKOzFQ_: Setting up session
[Instagram] DLuZNKOzFQ_: Downloading JSON metadata
[info] DLuZNKOzFQ_: Downloading 1 format(s): 1
[download] Sleeping 4.90 seconds ...
[download] Destination: downloaded_videos\video_1.mp4
[download] 100% of   14.97MiB in 00:00:00 at 17.83MiB/s  
✅ Success: video_1.mp4 (15.0 MB)
⏳ Waiting 3s...

📥 [2/2] https://www.instagram.com/reel/DM0ChALScuv/?igsh=MWphajVsYzFvMWcydg%3D%3D
[Instagram] Extracting URL: https://www.instagram.com/reel/DM0ChALScuv/?igsh=MWphajVsYzFvMWcydg%3D%3D
[Instagram] DM0ChALScuv: Setting up session
[Instagram] DM0ChALScuv: Downloading JSON metadata


ERROR: [Instagram] DM0ChALScuv: Restricted Video: You must be 18 years old or over to see this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies


❌ Error downloading https://www.instagram.com/reel/DM0ChALScuv/?igsh=MWphajVsYzFvMWcydg%3D%3D: ERROR: [Instagram] DM0ChALScuv: Restricted Video: You must be 18 years old or over to see this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies
❌ Failed
⏳ Waiting 3s...


🎉 DONE
